In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from skimage.measure import regionprops_table
import tifffile
from pathlib import Path
from tqdm import tqdm

In [2]:
BASE_PATH = Path(r"D:\Charlotte\CC_E19\Overall Segmentation")
subdirectories = [d for d in BASE_PATH.glob("*") if d.is_dir()]

all_all_dfs = {}

for subdir in subdirectories[0:1]:
    raw_tif_file = list(subdir.glob("*.tif"))[0]
    cell_line = raw_tif_file.stem.split(" ")[1]
    print(cell_line)

    raw_data = tifffile.imread(raw_tif_file)

    cell_masks_file = list((subdir / "Full Cells" / "masks").glob("*.tif"))[0]
    cell_masks = tifffile.imread(cell_masks_file)

    #nuclei_masks_file = list((subdir / "Nuclei" / "masks").glob("*.tif"))[0]
    #nuclei_masks = tifffile.imread(nuclei_masks_file)

    all_dfs = []

    for frame in tqdm(range(cell_masks.shape[0])):
        cell_frame = cell_masks[frame]

        df = regionprops_table(cell_frame, properties = (
            "label", "area", "intensity_mean", "centroid"
        ))

        df = pd.DataFrame(df, index=df["label"])

        df.rename(
            columns = {
                "area": "nuclei_area_px",
                "intensity_mean-0": "red",
                "intensity_mean-1": "green",
                "intensity_mean-2": "blue",
                "centroid-0": "y_px",
                "centroid-1": "x_px",
            },

            inplace=True
        )

        df["frame"] = frame

        all_dfs.append(df)

    all_cells = pd.concat(all_dfs, ignore_index=True)
    all_all_dfs[raw_tif_file.stem] = all_cells

pYtag


  0%|          | 0/133 [00:00<?, ?it/s]


AttributeError: Attribute 'intensity_mean' unavailable when `intensity_image` has not been specified.